# Two agents completing a task via A2A, with the supervisor delegating to a specialist

**Outcome:** two agents completing a task via A2A, with the supervisor delegating to a specialist.

**Idea in one line:** the supervisor agent *discovers* a specialist through its **Agent Card**, decides whether to delegate, sends it a **Message** over A2A, and gets back a finished **Task** with an **Artifact**. The two agents share nothing except those A2A messages.

```
 user
  |
 [ Supervisor agent ]  (its model decides: answer myself, or delegate?)
  |   1. GET  /.well-known/agent-card.json  ---------------->  [ Pricing specialist agent ]
  |   2. POST SendMessage (JSON-RPC)  ---------------------->    own model + own price data
  |   3. <-- result.task: TASK_STATE_COMPLETED + artifact ----    (a separate A2A server)
  v
 final answer
```

| Term | Meaning |
|---|---|
| **Agent Card** | The specialist's public "business card": name, skills, and the URL to send messages to |
| **Message** | One turn of conversation, made of **parts** (here: text) |
| **Task** | The unit of work the specialist creates for a message; it has a **state** |
| **Artifact** | The result the specialist hands back (here: the price quote) |

**A2A vs MCP:** MCP connects an agent to *tools and data* (demo 3). A2A connects an agent to *another agent* that has its own reasoning. The supervisor treats the specialist as a black box.

**What is real here:** two agents talking over real HTTP, using the A2A v1.0 JSON-RPC binding (Agent Card, `SendMessage`, Task states, version header, error codes).
**What is simplified:** the A2A parts are written by hand with the standard library, so you can see exactly what goes over the wire. In production you would use the official `a2a-sdk`. Not covered: streaming, task polling (`GetTask`), push notifications and authentication.

## 0. Setup
No installs are needed for the mock run (standard library only).

**To try a real model** (after the mock run passes):
1. `pip install openai`, install [Ollama](https://ollama.com), and run `ollama pull qwen2.5:7b` (keep Ollama running).
2. In the settings cell below, set `BACKEND = "real"` and re-run **only that cell and the test cell in section 5**. Both agents then use the real model.
3. Small models can slip (wrong format, ignoring a rule). Tests 1, 6, 7 and 8 are plain protocol checks and do not depend on the model. Tests 2 to 5 use the model (the specialist's, and for 3 to 5 the supervisor's too).

In [ ]:
import json, re, threading, urllib.request, uuid
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

# ---- The ONLY settings you change to switch models ----
BACKEND  = "mock"                              # "mock" = fake model, "real" = a real model
BASE_URL = "http://localhost:11434/v1"         # Ollama (free, local). Any OpenAI-compatible URL works
API_KEY  = "ollama"
MODEL    = "qwen2.5:7b"

# ---- Where the specialist agent will listen ----
SPECIALIST_PORT = 9101
SPECIALIST_URL  = f"http://127.0.0.1:{SPECIALIST_PORT}"
A2A_VERSION     = "1.0"

## 1. The model: mock or real (each agent uses it as its own brain)

In [ ]:
def mock_model(prompt):
    """A fake model. It recognises which agent is asking from the prompt and replies with one line."""
    if "Customer request:" in prompt:                              # --- the SPECIALIST asking
        req = prompt.split("Customer request:")[1].split("\n")[0].lower()
        for product, keyword in (("laptop_stand", "laptop stand"), ("keyboard", "keyboard"), ("usb_cable", "usb cable")):
            if keyword in req:
                qty = re.search(r"\d+", req)
                return f"{product} {qty.group(0) if qty else 1}"
        return "UNKNOWN"
    req = prompt.split("User request:")[1].split("\n")[0].lower()  # --- the SUPERVISOR asking
    if re.search(r"price|cost|quote|how much|total", req):
        return "DELEGATE pricing-agent"
    return "ANSWER Hi! I'm the help-desk supervisor. Ask me for a price quote and I'll get it from our pricing specialist."

def real_model(prompt):
    """Ask a real model. Works with any OpenAI-compatible endpoint."""
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    reply = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return reply.choices[0].message.content

def ask_model(prompt):
    """The ONE place where mock vs real is decided."""
    return mock_model(prompt) if BACKEND == "mock" else real_model(prompt)

## 2. The specialist agent (an A2A server)
It does three things:
1. **Publishes an Agent Card** at `/.well-known/agent-card.json`, so others can discover what it does and where to send messages.
2. **Accepts `SendMessage`** (JSON-RPC 2.0 over HTTP), checks it, and creates a **Task**.
3. **Does the work**: its own model call works out the product and quantity, and its own code does the price maths. It returns the quote as an **Artifact**.

In [ ]:
# ---------- the specialist's own data and logic (the supervisor never sees any of this) ----------
CATALOGUE = {"laptop_stand": Decimal("29.99"), "keyboard": Decimal("49.00"), "usb_cable": Decimal("9.50")}
TAX = Decimal("0.08")

class QuoteError(Exception):
    pass

def price_quote(text):
    """The specialist's work: its model understands the request, its code does the maths."""
    prompt = f"""You are a pricing specialist. Work out which product and how many the customer wants.
Products: laptop_stand, keyboard, usb_cable
Customer request: {text}
Reply with exactly one line: <product> <quantity>, or UNKNOWN if the product is not in the list.
Answer:"""
    reply = ask_model(prompt)
    m = None if "UNKNOWN" in reply.upper() else re.search(r"\b(laptop_stand|keyboard|usb_cable)\b(?:\D{0,30}?(\d+))?", reply)
    if not m:
        raise QuoteError("I can only quote laptop_stand, keyboard and usb_cable.")
    product, qty = m.group(1), int(m.group(2) or 1)
    if not 1 <= qty <= 1000:
        raise QuoteError("Quantity must be between 1 and 1000.")
    subtotal = CATALOGUE[product] * qty
    tax = (subtotal * TAX).quantize(Decimal("0.01"), ROUND_HALF_UP)
    return f"{qty} x {product.replace('_', ' ')} @ ${CATALOGUE[product]} = ${subtotal}; tax (8%) ${tax}; total ${subtotal + tax}"

# ---------- the A2A part ----------
def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def agent_card():
    """The Agent Card: who I am, what I can do (skills), and where to send A2A messages."""
    return {
        "name": "pricing-agent",
        "description": "Quotes prices, including tax, for catalogue products (laptop_stand, keyboard, usb_cable).",
        "version": "1.0.0",
        "supportedInterfaces": [{"url": f"{SPECIALIST_URL}/a2a", "protocolBinding": "JSONRPC", "protocolVersion": A2A_VERSION}],
        "capabilities": {"streaming": False, "pushNotifications": False},
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
        "skills": [{"id": "price-quote", "name": "Price quote", "tags": ["pricing", "quote"],
                    "description": "Quote the price of a quantity of a catalogue product, including 8% tax.",
                    "examples": ["How much for 3 keyboards?"]}],
    }

def rpc_error(rid, code, message):
    return {"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": message}}

def message_text(msg):
    """Validate an incoming A2A message and return its text (or None if it is malformed)."""
    if not isinstance(msg, dict) or msg.get("role") != "ROLE_USER" or not msg.get("messageId"):
        return None
    parts = msg.get("parts")
    texts = [p["text"] for p in parts if isinstance(p, dict) and isinstance(p.get("text"), str)] if isinstance(parts, list) else []
    return " ".join(texts) or None

def run_task(msg, text):
    """Create a Task for the message, do the work, and report the final state."""
    task = {"id": str(uuid.uuid4()), "contextId": msg.get("contextId") or str(uuid.uuid4()),
            "status": {"state": "TASK_STATE_WORKING", "timestamp": now()}, "history": [msg]}
    try:
        quote = price_quote(text)
        task["status"] = {"state": "TASK_STATE_COMPLETED", "timestamp": now()}
        task["artifacts"] = [{"artifactId": str(uuid.uuid4()), "name": "quote", "parts": [{"text": quote}]}]
    except QuoteError as e:
        task["status"] = {"state": "TASK_STATE_FAILED", "timestamp": now(),
                          "message": {"messageId": str(uuid.uuid4()), "role": "ROLE_AGENT", "parts": [{"text": str(e)}]}}
    return task

RECEIVED = []        # every JSON-RPC method the specialist received (evidence for the tests)

def handle_rpc(req, version):
    rid = req.get("id") if isinstance(req, dict) else None
    try:
        if not isinstance(req, dict) or req.get("jsonrpc") != "2.0":
            return rpc_error(rid, -32600, "Invalid request")
        RECEIVED.append(req.get("method"))
        if version != A2A_VERSION:                                   # clients MUST send the A2A-Version header
            return rpc_error(rid, -32009, f"Version not supported: this agent speaks A2A {A2A_VERSION}")
        if req.get("method") != "SendMessage":
            return rpc_error(rid, -32601, f"Method not found: {req.get('method')}")
        msg = (req.get("params") or {}).get("message")
        text = message_text(msg)
        if text is None:
            return rpc_error(rid, -32602, "Invalid params: need a ROLE_USER message with a messageId and a text part")
        return {"jsonrpc": "2.0", "id": rid, "result": {"task": run_task(msg, text)}}
    except Exception as e:                                           # e.g. the model is unreachable
        return rpc_error(rid, -32603, f"Internal error: {type(e).__name__}: {e}")

class A2AHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):          # keep the notebook output quiet
        pass

    def _send(self, obj, code=200):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/.well-known/agent-card.json":
            self._send(agent_card())
        else:
            self._send({"error": "not found"}, 404)

    def do_POST(self):
        if self.path != "/a2a":
            return self._send({"error": "not found"}, 404)
        raw = self.rfile.read(int(self.headers.get("Content-Length", 0)))
        try:
            req = json.loads(raw)
        except ValueError:
            return self._send(rpc_error(None, -32700, "Parse error"))
        self._send(handle_rpc(req, self.headers.get("A2A-Version")))

# ---------- run it as a separate server ----------
specialist_server = None

def stop_specialist():
    global specialist_server
    if specialist_server is not None:
        specialist_server.shutdown(); specialist_server.server_close(); specialist_server = None
        print("Pricing specialist stopped.")

def start_specialist():
    global specialist_server
    stop_specialist()                                         # safe to re-run this cell
    specialist_server = ThreadingHTTPServer(("127.0.0.1", SPECIALIST_PORT), A2AHandler)
    threading.Thread(target=specialist_server.serve_forever, daemon=True).start()
    print("Pricing specialist (A2A server) listening at", SPECIALIST_URL)

start_specialist()

## 3. The supervisor agent (an A2A client)
It only knows the specialist's **URL**. Steps: discover the Agent Card, let its model decide *answer myself or delegate*, and if delegating, send an A2A message and read the Task that comes back.

In [ ]:
SPECIALIST_URLS = [SPECIALIST_URL]            # add more specialists here and they are discovered automatically

# A direct connection: ignore any system proxy settings for localhost traffic
opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def get_json(url):
    with opener.open(url, timeout=10) as r:
        return json.load(r)

def rpc_post(url, payload, version=A2A_VERSION):
    """Send one JSON-RPC request. Returns the parsed JSON-RPC response."""
    headers = {"Content-Type": "application/json"}
    if version: headers["A2A-Version"] = version
    req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers=headers, method="POST")
    with opener.open(req, timeout=60) as r:
        return json.load(r)

def a2a_send_text(rpc_url, text):
    """Build a standard A2A message with one text part and send it with SendMessage."""
    message = {"messageId": str(uuid.uuid4()), "role": "ROLE_USER", "parts": [{"text": text}]}
    return rpc_post(rpc_url, {"jsonrpc": "2.0", "id": 1, "method": "SendMessage", "params": {"message": message}})

def discover(urls):
    """Step 1: fetch each agent's Agent Card and note where to send messages."""
    registry = {}
    for url in urls:
        card = get_json(url + "/.well-known/agent-card.json")
        iface = next(i for i in card["supportedInterfaces"] if i["protocolBinding"] == "JSONRPC")
        registry[card["name"]] = {"card": card, "rpc_url": iface["url"]}
    return registry

def run_supervisor(request):
    registry = discover(SPECIALIST_URLS)
    agents = "\n".join(f"- {name}: {a['card']['description']} (skills: {', '.join(s['id'] for s in a['card']['skills'])})"
                       for name, a in registry.items())
    prompt = f"""You are a help-desk supervisor agent. Answer simple questions yourself.
For anything about prices, costs or quotes you MUST delegate to a specialist; never guess prices.
Specialist agents (discovered from their A2A Agent Cards):
{agents}
User request: {request}
Reply with exactly one line: DELEGATE <agent_name>   or   ANSWER <your short answer>
Reply:"""
    reply = ask_model(prompt)                                        # Step 2: my model decides
    m = re.search(r"DELEGATE\s+([\w-]+)", reply)
    if not (m and m.group(1) in registry):                           # no delegation: answer myself
        return {"delegated": False, "task": None, "answer": re.sub(r"^\s*ANSWER\s*:?\s*", "", reply, flags=re.I).strip()}

    name = m.group(1)
    print(f"  supervisor: delegating to '{name}' via A2A SendMessage")
    resp = a2a_send_text(registry[name]["rpc_url"], request)         # Step 3: A2A call
    if "error" in resp:
        return {"delegated": True, "task": None, "answer": f"Sorry, {name} returned an error: {resp['error']['message']}"}
    task = resp["result"]["task"]
    state = task["status"]["state"]
    print(f"  {name}: task {task['id'][:8]} finished as {state}")
    if state == "TASK_STATE_COMPLETED":                              # Step 4: read the artifact
        answer = f"Here is your quote from our pricing specialist: {task['artifacts'][0]['parts'][0]['text']}"
    else:
        reason = task["status"].get("message", {}).get("parts", [{}])[0].get("text", "no reason given")
        answer = f"Sorry, the pricing specialist could not complete this: {reason}"
    return {"delegated": True, "task": task, "answer": answer}

## 4. See the raw A2A messages

In [ ]:
card = get_json(SPECIALIST_URL + "/.well-known/agent-card.json")
print("--- Agent Card (GET /.well-known/agent-card.json) ---")
print(json.dumps(card, indent=2))

message = {"messageId": str(uuid.uuid4()), "role": "ROLE_USER", "parts": [{"text": "How much for 3 laptop stands?"}]}
request = {"jsonrpc": "2.0", "id": 1, "method": "SendMessage", "params": {"message": message}}
print("\n--- Request: POST /a2a   (header A2A-Version: 1.0) ---")
print(json.dumps(request, indent=2))
print("\n--- Response ---")
print(json.dumps(rpc_post(card["supportedInterfaces"][0]["url"], request), indent=2))

## 5. Tests (work with either backend)

In [ ]:
def check(name, condition):
    print(("PASS  " if condition else "FAIL  ") + name)
    assert condition, name

print("Backend:", BACKEND, "\n")
RPC = SPECIALIST_URL + "/a2a"

# 1. Discovery: the supervisor can find the specialist and read what it offers
registry = discover(SPECIALIST_URLS)
card = registry["pricing-agent"]["card"]
check("1. Agent Card discovered: skill 'price-quote', JSON-RPC interface",
      card["skills"][0]["id"] == "price-quote" and registry["pricing-agent"]["rpc_url"] == RPC)

# 2. The A2A call itself: SendMessage returns a completed Task with the quote as an artifact
resp = a2a_send_text(RPC, "How much for 3 laptop stands?")
task = resp["result"]["task"]
print("   artifact:", task["artifacts"][0]["parts"][0]["text"])
check("2. SendMessage returned a TASK_STATE_COMPLETED task with the right quote",
      task["status"]["state"] == "TASK_STATE_COMPLETED" and "total $97.17" in task["artifacts"][0]["parts"][0]["text"])

# 3. The whole flow: the supervisor delegates a pricing question to the specialist
print("\nSupervisor: 'How much would 2 keyboards cost?'")
RECEIVED.clear()
out = run_supervisor("How much would 2 keyboards cost?")
print("  final answer:", out["answer"])
check("3. supervisor delegated, and the specialist received exactly one SendMessage",
      out["delegated"] and RECEIVED == ["SendMessage"])
check("3b. the final answer contains the specialist's quote ($105.84)", "total $105.84" in out["answer"])

# 4. The supervisor answers simple questions itself, with no A2A call at all
print("\nSupervisor: 'Hello, who are you?'")
RECEIVED.clear()
out = run_supervisor("Hello, who are you?")
print("  final answer:", out["answer"])
check("4. no delegation for a simple question, and the specialist was never called",
      (not out["delegated"]) and RECEIVED == [])

# 5. The specialist cannot help: a FAILED task is reported cleanly, not a crash
print("\nSupervisor: 'How much for a flying carpet?'")
out = run_supervisor("How much for a flying carpet?")
print("  final answer:", out["answer"])
check("5. unknown product -> task FAILED, supervisor reports it politely",
      out["delegated"] and out["task"]["status"]["state"] == "TASK_STATE_FAILED" and "could not" in out["answer"])

# 6. Protocol rule: a request without the A2A-Version header is refused (VersionNotSupportedError)
msg = {"messageId": "m1", "role": "ROLE_USER", "parts": [{"text": "How much for a keyboard?"}]}
resp = rpc_post(RPC, {"jsonrpc": "2.0", "id": 7, "method": "SendMessage", "params": {"message": msg}}, version=None)
check("6. missing A2A-Version header is rejected (error -32009)", resp["error"]["code"] == -32009 and resp["id"] == 7)

# 7. Unknown method
resp = rpc_post(RPC, {"jsonrpc": "2.0", "id": 8, "method": "DoSomethingElse", "params": {}})
check("7. unknown method is rejected (error -32601)", resp["error"]["code"] == -32601)

# 8. Malformed message (no parts)
resp = rpc_post(RPC, {"jsonrpc": "2.0", "id": 9, "method": "SendMessage",
                      "params": {"message": {"messageId": "m2", "role": "ROLE_USER", "parts": []}}})
check("8. message without parts is rejected (error -32602)", resp["error"]["code"] == -32602)

print("\nOutcome met: the supervisor discovered the specialist, delegated over A2A, and the specialist completed the task.")

## 6. Clean up

In [ ]:
stop_specialist()